In [2]:
# Carregar 2 datasets:
# HaluEval — respostas com hallucinations já identificadas
# amnesty_qa — perguntas com contextos reais (perfeito para RAG)

from datasets import load_dataset

# Dataset 1
ds_halu = load_dataset("pminervini/HaluEval", "qa_samples",
                        split="data", streaming=True)

# Dataset 2
ds_rag = load_dataset(
    "explodinggradients/amnesty_qa",
    "english_v1",
    split="eval"
)

print("✅ Datasets carregados")

eval.parquet:   0%|          | 0.00/47.5k [00:00<?, ?B/s]

Generating eval split: 0 examples [00:00, ? examples/s]

✅ Datasets carregados


In [9]:
# Ver como são os dados de hallucination
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Pegar 20 exemplos para explorar
it = iter(ds_halu)
amostras = [next(it) for _ in range(20)]

print("Colunas:", df_halu.columns.tolist())
print(f"Exemplos carregados: {len(df_halu)}")
df_halu.head(3)

Colunas: ['knowledge', 'question', 'answer', 'hallucination']
Exemplos carregados: 20


,knowledge,question,answer,hallucination
0,Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.,Which magazine was started first Arthur's Magazine or First for Women?,First for Women was started first.,yes
1,Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.,Which magazine was started first Arthur's Magazine or First for Women?,First for Women was started first.,yes
2,Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.,Which magazine was started first Arthur's Magazine or First for Women?,First for Women was started first.,yes


In [10]:
import pprint

pprint.pprint(df_halu.iloc[0].to_dict())

{'answer': 'First for Women was started first.',
 'hallucination': 'yes',
 'knowledge': "Arthur's Magazine (1844–1846) was an American literary "
              'periodical published in Philadelphia in the 19th century.First '
              "for Women is a woman's magazine published by Bauer Media Group "
              'in the USA.',
 'question': "Which magazine was started first Arthur's Magazine or First for "
             'Women?'}


In [11]:
# Ver como são os dados RAG
df_rag = pd.DataFrame(ds_rag)

print("Colunas:", df_rag.columns.tolist())
print(f"Total de exemplos: {len(df_rag)}")
df_rag.head(3)

Colunas: ['question', 'ground_truths', 'answer', 'contexts']
Total de exemplos: 20


,question,ground_truths,answer,contexts
0,What are the global implications of the USA Supreme Court ruling on abortion?,"[The global implications of the USA Supreme Court ruling on abortion are significant. The ruling has led to limited or no access to abortion for one in three women and girls of reproductive age in states where abortion access is restricted. These states also have weaker maternal health support, higher maternal death rates, and higher child poverty rates. Additionally, the ruling has had an impact beyond national borders due to the USA's geopolitical and cultural influence globally. Organizations and activists worldwide are concerned that the ruling may inspire anti-abortion legislative and policy attacks in other countries. The ruling has also hindered progressive law reform and the implementation of abortion guidelines in certain African countries. Furthermore, the ruling has created a chilling effect in international policy spaces, empowering anti-abortion actors to undermine human rights protections.]","The global implications of the USA Supreme Court ruling on abortion can be significant, as it sets a precedent for other countries and influences the global discourse on reproductive rights. Here are some potential implications:\n\n1. Influence on other countries: The Supreme Court's ruling can serve as a reference point for other countries grappling with their own abortion laws. It can provide legal arguments and reasoning that advocates for reproductive rights can use to challenge restrictive abortion laws in their respective jurisdictions.\n\n2. Strengthening of global reproductive rights movements: A favorable ruling by the Supreme Court can energize and empower reproductive rights movements worldwide. It can serve as a rallying point for activists and organizations advocating for women's rights, leading to increased mobilization and advocacy efforts globally.\n\n3. Counteracting anti-abortion movements: Conversely, a ruling that restricts abortion rights can embolden anti-abortion movements globally. It can provide legitimacy to their arguments and encourage similar restrictive measures in other countries, potentially leading to a rollback of existing reproductive rights.\n\n4. Impact on international aid and policies: The Supreme Court's ruling can influence international aid and policies related to reproductive health. It can shape the priorities and funding decisions of donor countries and organizations, potentially leading to increased support for reproductive rights initiatives or conversely, restrictions on funding for abortion-related services.\n\n5. Shaping international human rights standards: The ruling can contribute to the development of international human rights standards regarding reproductive rights. It can influence the interpretation and application of existing human rights treaties and conventions, potentially strengthening the recognition of reproductive rights as fundamental human rights globally.\n\n6. Global health implications: The Supreme Court's ruling can have implications for global health outcomes, particularly in countries with restrictive abortion laws. It can impact the availability and accessibility of safe and legal abortion services, potentially leading to an increase in unsafe abortions and related health complications.\n\nIt is important to note that the specific implications will depend on the nature of the Supreme Court ruling and the subsequent actions taken by governments, activists, and organizations both within and outside the United States.","[- In 2022, the USA Supreme Court handed down a decision ruling that overturned 50 years of jurisprudence recognizing a constitutional right to abortion.\n- This decision has had a massive impact: one in three women and girls of reproductive age now live in states where abortion access is either totally or near-totally inaccessible.\n- The states with the most restrictive abortion laws have the weakest maternal health s

In [12]:
pprint.pprint(df_rag.iloc[0].to_dict())

{'answer': 'The global implications of the USA Supreme Court ruling on '
           'abortion can be significant, as it sets a precedent for other '
           'countries and influences the global discourse on reproductive '
           'rights. Here are some potential implications:\n'
           '\n'
           "1. Influence on other countries: The Supreme Court's ruling can "
           'serve as a reference point for other countries grappling with '
           'their own abortion laws. It can provide legal arguments and '
           'reasoning that advocates for reproductive rights can use to '
           'challenge restrictive abortion laws in their respective '
           'jurisdictions.\n'
           '\n'
           '2. Strengthening of global reproductive rights movements: A '
           'favorable ruling by the Supreme Court can energize and empower '
           'reproductive rights movements worldwide. It can serve as a '
           'rallying point for activists and organizatio

# O que podemos observar nos dados:

# 1. Quais colunas existem em cada dataset? 
Colunas no df_halu: ['knowledge', 'question', 'answer', 'hallucination'];
Colunas no df_rag: ['question', 'ground_truths', 'answer', 'contexts']

# 2. Como é uma resposta com hallucination vs. sem?
A resposta com hallucination não responde exatamente o que a pergunta pede, porém tem informação dentro do contexto da pergunta, mas é mais generalizado e não específico.

# 3. O amnesty_qa tem contextos (coluna 'contexts')? Como são?
Sim, tem a coluna 'contexts', elas descrevem o cenário de acordo com a pergunta feita, é um panorama detalhado em que se insere a pergunta.

